# Multi-label support ticket pipeline (end-to-end)

This notebook drives the **modular** package in `ml_pipeline/`.

**Task:** multi-label classification — each ticket is tagged with **`category:*`** and **`priority:*`** labels (two active labels in a shared vocabulary). This replaces the old single-label `LabelEncoder` setup.

**Critical fixes vs. the previous notebook:**
- `MultiLabelBinarizer` fit **only on training** data (no label leakage).
- Train / validation / test split; **test is never used for early stopping** or model selection.
- **No outcome columns** (resolution times, satisfaction, SLA, etc.) in features — leakage guard in `build_preprocessed_frame`.
- **Inference uses the same** `clean_ticket_text` as training (via the package).
- **Multi-label metrics:** micro/macro F1, Hamming loss, precision/recall, subset accuracy.
- **Sklearn `Pipeline`:** TF-IDF text + scaled numerics + one-hot categoricals + `OneVsRest(LogisticRegression)`.
- **Cross-validation** on the training set only.
- Optional **DistilBERT** path with `problem_type="multi_label_classification"` (sigmoid + BCE).

Install dependencies: `pip install -r requirements.txt`

In [ ]:
from pathlib import Path
import sys

# Repo root on sys.path (run notebook from project root)
ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd

from ml_pipeline.config import DEFAULT_DATA_PATH, SplitConfig, TransformerTrainConfig
from ml_pipeline.features import modeling_feature_columns
from ml_pipeline.metrics import multilabel_metric_bundle
from ml_pipeline.preprocessing import (
    build_preprocessed_frame,
    load_raw_tickets,
    make_train_val_test,
)
from ml_pipeline.model_io import load_sklearn_bundle, predict_sklearn_multilabel, save_sklearn_bundle
from ml_pipeline.sklearn_pipeline import (
    build_classifier_chain_pipeline,
    build_sklearn_multilabel_pipeline,
    run_cross_validation,
)
from ml_pipeline.transformer_multilabel import (
    evaluate_multilabel_transformer_on_test,
    load_multilabel_transformer,
    train_multilabel_transformer,
)

In [ ]:
# --- User knobs ---
NROWS = 25_000  # None = full CSV (slow / memory-heavy)
CV_SPLITS = 3
DATA_PATH = DEFAULT_DATA_PATH
split_cfg = SplitConfig(test_size=0.15, val_size=0.15, random_state=42)

# Saved artifacts (under project root)
SKLEARN_ARTIFACT_DIR = ROOT / "outputs" / "sklearn_multilabel"
TRANSFORMER_DIR = ROOT / "outputs" / "distilbert_multilabel"

# Transformer: train / resume Trainer checkpoint / evaluate disk model
RUN_TRANSFORMER = False  # True: run HF fine-tuning (or resume)
RESUME_TRANSFORMER = None  # None | True (latest in output_dir) | path to checkpoint-* folder
EVAL_SAVED_TRANSFORMER = True  # After training, or alone: load TRANSFORMER_DIR and score test set

## 1) Load, clean, multi-label targets, split

- Missing text or empty label sets are dropped.
- `issue_description` column is overwritten with **cleaned** text for the sklearn column name contract.
- Tabular imputation + scaling happens **inside** the sklearn `Pipeline` (fit on train only).

In [ ]:
df_raw = load_raw_tickets(DATA_PATH, nrows=NROWS)
df = build_preprocessed_frame(df_raw, drop_duplicate_tickets=True)

train_df, val_df, test_df, mlb, Y_train, Y_val, Y_test = make_train_val_test(
    df, mlb=None, split=split_cfg, stratify_key="category"
)

print("Rows — train:", len(train_df), "val:", len(val_df), "test:", len(test_df))
print("Label space size:", len(mlb.classes_))
print("Sample label sets:", train_df["label_set"].head(3).tolist())

## 2) Sklearn baseline — Pipeline + cross-validation + final evaluation

- CV scores are **only** on training data folds.
- After CV, we **refit** on the full training split and report **validation** and **test** metrics.
- **Test** metrics should be reported **once** for unbiased generalization estimates.

In [ ]:
feat_cols = modeling_feature_columns(train_df)
X_train = train_df[feat_cols]
X_val = val_df[feat_cols]
X_test = test_df[feat_cols]

sk_model = build_sklearn_multilabel_pipeline(X_layout=X_train)

cv_summary = run_cross_validation(sk_model, X_train, Y_train, cv_splits=CV_SPLITS)
print("CV F1 micro:", cv_summary)

sk_model.fit(X_train, Y_train)

val_pred = sk_model.predict(X_val)
test_pred = sk_model.predict(X_test)

print("\n=== Validation (for tuning) ===")
print(multilabel_metric_bundle(Y_val, val_pred))
print("\n=== Test (report once) ===")
print(multilabel_metric_bundle(Y_test, test_pred))

In [ ]:
# Save sklearn Pipeline + MultiLabelBinarizer + feature column order (reuse without retraining)
SKLEARN_ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
save_sklearn_bundle(SKLEARN_ARTIFACT_DIR, sk_model, mlb, feat_cols)
print("Saved sklearn bundle to", SKLEARN_ARTIFACT_DIR.resolve())

# Optional reload smoke test:
# pipe, mlb_loaded, cols = load_sklearn_bundle(SKLEARN_ARTIFACT_DIR)
# _, decoded = predict_sklearn_multilabel(pipe, mlb_loaded, cols, X_test.head(3))
# print(decoded)

## 3) Optional — Classifier chain (label correlation)

Uncomment to compare with `OneVsRest`; chains can help when labels are correlated but train more slowly.

In [ ]:
# chain = build_classifier_chain_pipeline(X_layout=X_train, order=None)
# chain.fit(X_train, Y_train)
# print(multilabel_metric_bundle(Y_test, chain.predict(X_test)))

## 4) DistilBERT multi-label — train, resume, save, evaluate

- **Train** with `RUN_TRANSFORMER=True` (downloads weights first run). Checkpoints + `label_names.json` live under `TRANSFORMER_DIR`.
- **Continue training**: set `RESUME_TRANSFORMER=True` or pass a `checkpoint-*` path.
- **Evaluate only**: train once, then run with `RUN_TRANSFORMER=False` and `EVAL_SAVED_TRANSFORMER=True` to load the saved model and compute test metrics (no leakage: test still held out).

In [ ]:
TRANSFORMER_DIR.mkdir(parents=True, exist_ok=True)
label_names = mlb.classes_.tolist()

tcfg = TransformerTrainConfig(
    num_train_epochs=2,
    output_dir=str(TRANSFORMER_DIR),
    fp16=True,  # applied only when CUDA is available (see ml_pipeline.transformer_multilabel)
)

if RUN_TRANSFORMER:
    trainer, tok = train_multilabel_transformer(
        train_df,
        val_df,
        Y_train,
        Y_val,
        text_column="clean_text",
        cfg=tcfg,
        label_names=label_names,
        resume_from_checkpoint=RESUME_TRANSFORMER,
    )
    print("Transformer training finished; artifacts at", TRANSFORMER_DIR.resolve())

ckpt_marker = TRANSFORMER_DIR / "config.json"
if ckpt_marker.is_file() and EVAL_SAVED_TRANSFORMER:
    hf_model, hf_tok = load_multilabel_transformer(TRANSFORMER_DIR)
    tx_metrics = evaluate_multilabel_transformer_on_test(
        hf_model,
        hf_tok,
        test_df["clean_text"].tolist(),
        Y_test,
        threshold=tcfg.prediction_threshold,
        max_length=tcfg.max_length,
    )
    print("Transformer — test set metrics (loaded model):", tx_metrics)
elif not ckpt_marker.is_file():
    print(
        "No saved transformer in",
        TRANSFORMER_DIR,
        "— set RUN_TRANSFORMER=True to train, or point TRANSFORMER_DIR to an existing checkpoint.",
    )